In [ ]:
import os
from pathlib import Path

SCRATCH = Path.home() / "scratch"
os.environ["HF_HOME"] = str(SCRATCH / "hf_home")

In [ ]:
import gc
import re
import time
from typing import Any, Dict, List, Tuple, Union

import deepspeed
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from deepspeed import DeepSpeedEngine
from tqdm import trange
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel
from vllm import LLM, SamplingParams

import wandb
from utils import (
	compute_token_log_probs,
#     dump_episodes,
#     evaluate_on_test_set,
#     find_free_port,
#     find_last_checkpoint,
#     prepare_model_inputs,
#     load_model_into_vllm
)

# Needed to stop DeepSpeed from complaining
os.environ["MASTER_ADDR"] = "localhost"
# os.environ["MASTER_PORT"] = str(find_free_port())
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

In [ ]:
# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_CHAT_NAME = MODEL_NAME + "-Instruct"

# Dataset configuration
DATASET_NAME = "EleutherAI/hendrycks_math"

In [ ]:
# Total number of training iterations
NUM_ITERATIONS = 1000
# Number of episodes to collect per iteration for training
EPISODES_PER_ITERATION = 64
# Number of responses to generate for each input prompt (i.e. group size in GRPO)
GENERATIONS_PER_SAMPLE = 4
# Controls how much the policy can deviate from the reference model
KL_COEFFICIENT = 0.001

# Training hyperparameters
# Batch size for each GPU device during training
PER_DEVICE_BATCH_SIZE = 4
# Learning rate for model updates
LEARNING_RATE = 1e-6

# Sampling parameters
# Maximum number of tokens to generate in each response
MAX_RESPONSE_TOKENS = 1024
# Controls randomness in generation (higher = more random)
TEMPERATURE = 1.0
# Nucleus sampling parameter (1.0 = disabled)
TOP_P = 1.0
# Top-k sampling parameter (-1 = disabled)
TOP_K = -1  # no top k

# DeepSpeed configuration
# DeepSpeed config for the policy model
deepspeed_config = {
	"bf16": {"enabled": True},
	"zero_optimization": {"stage": 2, "overlap_comm": False},
	"train_batch_size": EPISODES_PER_ITERATION, # optimizer step
	"train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE, # per GPU allocated batch size
	"gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE, # increase our effective batch size by adding up gradients across devices
	"gradient_clipping": 1.0,
	"optimizer": {
		"type": "AdamW",
		"params": {
			"lr": LEARNING_RATE,
			"betas": (0.9, 0.999),
			"eps": 1e-8,
			"weight_decay": 0.0,
			"torch_adam": True,
		},
	},
}
# DeepSpeed config for the reference model that we use to compute KL divergence
ref_deepspeed_config = {
	"bf16": {"enabled": True},
	# Note that we don't train the reference model
	# These are just for compatibility with DeepSpeed.
	"train_batch_size": EPISODES_PER_ITERATION,
	"train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE,
	"gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE,
}

RUN_NAME = "r1-zero"
EXP_DIR = SCRATCH / "deepseek_r1z_hackathon" / RUN_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Logs and Checkpoints will be saved to: {EXP_DIR}")

Logs and Checkpoints will be saved to: /Users/jackzheng/scratch/deepseek_r1z_hackathon/r1-zero


In [ ]:
SYSTEM_MESSAGE = (
	"You are a helpful assistant. You first think about the reasoning process in the mind "
	"and then provide the user with the answer."
)
PROMPT_TEMPLATE = (
	"{problem}\n"
	"Show your work in <think> </think> tags. And return the final answer in "
	"<answer> </answer> tags, for example <answer>42</answer>."
)

In [ ]:
# Extract solution 

def extract_boxed_answer(solution: str): 

	idx = solution.rfind(r"\boxed") # reversed find

	if idx == -1: 
		return None # didn't find solution, return -1
	
	i = idx+len(r"\boxed")
	while i < len(solution) and solution[i] != "{":
		i += 1 # example solution: \boxed{0}$. We are just trying to skip whitespacae
	if i >= len(solution): 
		return None
	num_open_left_brace = 0
	start = i + 1 
	for j in range(i, len(solution)): 
		if solution[j] == "{":
			num_open_left_brace += 1 # counting the open brackets, ensuring they are closed for cases like \boxed   {\dfrac{1}{2}}
		if solution[j] == "}":
			num_open_left_brace -= 1
			if num_open_left_brace == 0:
				return solution[start:j].strip()
	return ""

Tokenizing + loading dataset


In [ ]:
# we use the chat model because it has apply chat template, which is the format we want to converse with this LLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHAT_NAME)
EOS_TOKEN_ID = AutoTokenizer.from_pretrained(MODEL_NAME).eos_token_id
EOS_TOKEN = tokenizer.convert_ids_to_tokens(EOS_TOKEN_ID)

# Dataloader to load MATH dataset
def data_loader(example: Dict[str, Any]):

	prefix = [
		{"role": "system", "content": SYSTEM_MESSAGE},
		{"role": "user", "content": PROMPT_TEMPLATE.format(problem=example["problem"])}, # load the problem set in
		{"role": "assistant", "content": "Let me solve this step by step.\n<think>"},
	]
	input_ids = tokenizer.apply_chat_template(
		prefix, tokenize=True, continue_final_message=True # we want to model to generate its own answer
	)
	prompt = tokenizer.decode(
		input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
	)
	return {"prompt": prompt, "input_ids": input_ids, "answer": extract_boxed_answer(example["solution"])}

# load the MATH dataset that contains problems across subjects
MATH_SUBJECTS = [
	 "algebra", "counting_and_probability", "geometry", "intermediate_algebra", "number_theory", "prealgebra", "precalculus",
]

def load_math_split(split: str): 
	data = [load_dataset(DATASET_NAME, name=s, split=split) for s in MATH_SUBJECTS] # assemble array 
	return concatenate_datasets(data)

train_dataset = load_math_split('train').map(data_loader).filter(lambda ex: ex["answer"] is not None)
test_dataset = load_math_split('test').map(data_loader).filter(lambda ex: ex["answer"] is not None)

Filter: 100%|██████████| 5000/5000 [00:00<00:00, 26124.76 examples/s]


In [ ]:
len(train_dataset), len(test_dataset)

train_dataset[0]

{'problem': 'Let \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &\\text{ if } -2 \\le x \\le 2, \\\\\n2x-b &\\text{ if } x <-2.\n\\end{array}\n\\right.\\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).',
 'level': 'Level 5',
 'type': 'Algebra',
 'solution': 'For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \\Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be equal when $x=-2$. Substituting, we get $-2-5=2(-2)-b$, which implies $b=3$. So $a+b=-3+3=\\boxed{0}$.',
 'prompt': '<|im_start|>system\nYou are a helpful assistant. You first think about the reasoning process in the mind and then provide the user with the answer.<|im_end|>\n<|im_start|>user\nLet \\[f(x) = \\left\\{\n\\begin{array}{cl} ax+3, &\\text{ if }x>2, \\\\\nx-5 &

Building the reward function


In [ ]:
from math_verify import parse, verify

def format_reward_fn(completion: str): 
	try:
		# add synthetic <think> as its already part of the prompt and prefilled 
		# for the assistant to more easily match the regex
		completion = "<think>" + completion

		# Strip EOS token if present
		if completion.endswith(EOS_TOKEN):
			completion = completion[:-len(EOS_TOKEN)]

		# Check if the format is correct
		# Pattern means:
		# 1) <think>...contents not including other <think> tags...</think>
		# 2) \n
		# 3) <answer>...anything...</answer>  
		regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"
		match = re.search(regex, completion, re.DOTALL)

		if match is None or len(match.groups()) != 2:
			# Format is incorrect
			return 0.0
		else:
			# Extract the content inside <answer>...</answer>
			answer_content = match.group(2).strip()

			# Check if answer content matches the allowed pattern
			if not re.match(allowed_pattern, answer_content):
				# If it doesn't match, reward is 0.5
				return 0.5
			# If both format and pattern are correct, reward is 1
			return 1.0
	except Exception:
		# Any error leads to 0 reward
		return 0.0

def answer_reward_fn(completion: str, prompt: str, answer: str): 

	try:
		# Pull the text inside the LAST <answer>...</answer> the model wrote.
		matches = re.findall(r"<answer>([\s\S]*?)<\/answer>", completion)
		if not matches:
			return 0.0
		model_answer = matches[-1].strip() # get the last answer match
		if model_answer == "" or answer is None: # if empty, return 0 reward
			return 0.0

		# math_verify parses both sides into symbolic math  objects and checks equivalence (so \frac{1}{2}, 0.5, and 2/4 all count as equal).
		# verify(gold, target) -> True if equal. We wrap the gold answer in $...$ so the parser treats it as LaTeX math.
		gold = parse(f"${answer}$")
		pred = parse(model_answer)
		return 1.0 if verify(gold, pred) else 0.0
	except Exception:
		return 0.0

def compute_reward(completion: str, sample: Dict[str, Any]):

	format_reward = format_reward_fn(completion)
	answer_reward = answer_reward_fn(completion, sample['prompt'], sample['answer'])
	reward = format_reward + answer_reward

	metrics = { # for graphing later on
		"format_reward": format_reward,
		"answer_reward": answer_reward,
		"reward": reward,
	}

	return reward, metrics

In [ ]:
# each response is a list of dictionaries, each dictionary containing {prompt: str, answer: str, etc.}
def calculate_training_rewards(sample: List[Dict[str: Any]], all_generations: List[List[int]], all_finish_reasons: List[str]):

	assert len(all_generations) == len(all_finish_reasons)
	assert len(all_generations) == len(samples) * GENERATIONS_PER_SAMPLE

	groups = [
		# list defined by 
		list(range(i, i + GENERATIONS_PER_SAMPLE)) for i in range(0, len(all_generations), GENERATIONS_PER_SAMPLE)
	]

	all_query_token_ids, all_responses_token_ids, all_advantages = [], [], []

	stats = {
		"response_lengths": [],
		"rewards": [],
		"non_stop_rate": [],
	}

	# sample is a list of dicts - the prompt, answer, etc. tuples, for which we've sampled multiple completions for 
	for sample, group_indices in zip(sample, group):
		# where group_indices is [1,2,3,4] if GENERATIONS_PER_SAMPLE = 4 for example
		finish_reasons = [all_finish_reasons[i] for i in group_indices] # pluck out reasons from that group
		response_token_ids = [all_generations[i] for i in group_indices] # pluck out response tokens, returns array
		responses = tokenzier.batch_decode(response_token_ids, skip_special_tokens=False) # returns array but decoded with [completion1, completion2, etc.]
		
		####################################
		print(responses[0])
		####################################

		metrics = [compute_reward(resp, sample) for resp,sample in zip(responses, sample)] # compare the response with the golden, return array of rewards for each
		rewards, reward_metrics = zip(*metrics) # will return as 'rewards, metrics'

		rewards = np.array(rewards)
		response_advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4) # normalize to get advantages per token

		advantages = [
			[resp_adv] * len(resp) for resp_adv, resp in zip(response_advantages, response_token_ids) # this is list multiplication
			# so we get [7,7,7,7,...] where the # of 7s is the length of our list, now we want to match those with the tokens
		]

		all_query_token_ids.extend([sample["input_ids"]] * GENERATIONS_PER_SAMPLE)
		all_responses_token_ids.extend(response_token_ids)
		all_advantages.extend(advantages)

		stats["rewards"].extend(rewards)
		stats["non_stop_rate"].extend([fr != "stop" for fr in finish_reasons])
		stats["response_lengths"].extend([len(ids) for ids in response_token_ids])
		for rm in reward_metrics:
			for k, v in rm.items():
				stats.setdefault(f"reward_metrics/{k}", []).append(v)

	episodes = {
		"all_query_token_ids": all_query_token_ids,
		"all_response_token_ids": all_responses_token_ids,
		"all_advantages": all_advantages,
	} 

	return episodes, stats

TabError: inconsistent use of tabs and spaces in indentation (<string>, line 21)

Tmrw change this to CISPO as a challenge


In [3]:
def compute_loss(
	policy_model: DeepSpeedEngine | PreTrainedModel
	reference_model: DeepSpeedEngine | PreTrainedModel,
	batch: Dict[str, torch.Tensor],
	total_response_len: int,
):
	input_ids = batch["input_ids"]  # [batch_size, seq_len]
	attention_mask = batch["attention_mask"]  # [batch_size, seq_len]
	labels = batch["labels"]  # [batch_size, seq_len]
	advantages = batch["advantages"]  # [batch_size, seq_len]

	model_inputs = {
		"input_ids": input_ids,
		"attention_mask": attention_mask,
		"labels": labels,
	}

	labels_mask = (labels[..., 1:] != -100).float()  # take the last dimension and grab only from first to last 
	# we want to compare position 0 to position 1, etc. - [batch_size, seq_len-1]
	# set all labels that are not equal to -100 to True, so that means we will train on them

	with torch.no_grad():
		ref_logps = compute_token_log_probs(
			reference_model, model_inputs, TEMPERATURE
		)  # [batch_size, seq_len-1]

	# need to recompute log_probs since we want to track gradients
	logps = compute_token_log_probs(policy_model, model_inputs, TEMPERATURE)  # [batch_size, seq_len-1]

	# 
	kl_penalty = torch.exp(ref_logps - logps) - (ref_logps - logps) - 1  # [batch_size, seq_len-1]
	kl_penalty = kl_penalty * labels_mask  # [batch_size, seq_len-1] element-wise multiply that zeros out the padding tokens

	entropy = -logps.sum() / labels_mask.sum()  # scalar

	policy_loss = -logps * advantages[..., 1:]  # [batch_size, seq_len-1]
	policy_loss = policy_loss * labels_mask  # [batch_size, seq_len-1]

	loss = (policy_loss + KL_COEFFICIENT * kl_penalty).sum() / total_response_len  # scalar

	metrics = {
		"policy_loss": policy_loss.sum().item() / total_response_len,
		"kl_penalty": kl_penalty.sum().item() / total_response_len,
		"entropy": entropy.item() / total_response_len,
	}

	return loss, metrics

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4287624569.py, line 2)